# E11 — A partilha do relógio

O capítulo anterior mediu o preço de mexer no mundo de propósito e fechou dizendo que o orçamento
é um só: os dias gastos segurando o sistema são dias em que ele não aprende. A pergunta deste
caderno é a travessia entre proteger e adaptar: **recalibrar a barreira compete com acompanhar o
mundo?**

A barreira que este livro sabe construir tem dois ingredientes. A **escala** é o nível da
oscilação, que muda depressa e é o que uma mudança de verdade mexe. A **forma** é o corte
padronizado, que diz quantos desvios acima da escala fica a barreira, e que muda devagar, porque
padronizar já tirou dele o que a escala carregava. A barreira do dia é o produto dos dois.

Cada ingrediente custa trabalho para ser refeito. O orçamento é contado em **atualizações ao longo
do horizonte**, e a perda é a que a barreira **não** segurou: a soma do que passou por cima dela,
dia a dia. Uma moeda só, para os dois ingredientes, de modo que a comparação não dependa de
declarar um câmbio entre coisas diferentes.


In [1]:
# <- brinque com: SEMENTES, ORCAMENTO, FRACOES, CICLOS, HORIZONTE, QUANDO
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import graficos, mudanca, partilha

SEMENTES = 20
DIAS = 12000
QUANDO = 3000
FATOR = 2.0
HORIZONTE = 500
ORCAMENTO = 60
FRACOES = (0.0, 0.25, 0.5, 0.75, 0.9, 1.0)
ORCAMENTOS = (5, 10, 20, 40, 60, 120, 250)
CICLOS = (1, 2, 5, 10, 21, 50, 126, 250)
SEMENTE = 400

dobra = [np.abs(mudanca.degrau(DIAS, np.random.default_rng(SEMENTE + i), fator=FATOR,
                                  quando=QUANDO)) for i in range(SEMENTES)]
parado = [np.abs(mudanca.degrau(DIAS, np.random.default_rng(SEMENTE + i), fator=1.0,
                                quando=QUANDO)) for i in range(SEMENTES)]
print("series: %d de %d dias | horizonte %d | janelas %d e %d | posto %d"
      % (SEMENTES, DIAS, HORIZONTE, partilha.JANELA_NIVEL, partilha.JANELA_CORTE, partilha.POSTO))


series: 20 de 12000 dias | horizonte 500 | janelas 21 e 252 | posto 13


In [2]:
# As duas pressas: cada ingrediente atualizado no seu ciclo, o outro todo dia.
linhas = []
for ciclo in CICLOS:
    m_n, d_n = partilha.perdas(dobra, ciclo, 1, inicio=QUANDO, horizonte=HORIZONTE)
    m_c, d_c = partilha.perdas(dobra, 1, ciclo, inicio=QUANDO, horizonte=HORIZONTE)
    linhas.append({"ciclo": ciclo, "escala": m_n, "escala_dp": d_n, "forma": m_c, "forma_dp": d_c})
prensas = pd.DataFrame(linhas).set_index("ciclo")
print(prensas.round(6).to_string())
piso = partilha.perdas(parado, 1, 1, inicio=QUANDO, horizonte=HORIZONTE)
print()
print("o controle, num mundo que nao muda: %.6f (dp %.6f)" % piso)
print("com tudo atualizado todo dia, no mundo que dobra: escala %.6f | forma %.6f"
      % (prensas.loc[1, "escala"], prensas.loc[1, "forma"]))


         escala  escala_dp     forma  forma_dp
ciclo                                         
1      0.000397   0.000082  0.000397  0.000082
2      0.000398   0.000080  0.000397  0.000082
5      0.000402   0.000082  0.000397  0.000081
10     0.000427   0.000091  0.000398  0.000083
21     0.000500   0.000119  0.000397  0.000086
50     0.000646   0.000175  0.000395  0.000084
126    0.001140   0.000422  0.000399  0.000098
250    0.001807   0.000657  0.000383  0.000096

o controle, num mundo que nao muda: 0.000200 (dp 0.000041)
com tudo atualizado todo dia, no mundo que dobra: escala 0.000397 | forma 0.000397


In [3]:
# A partilha: o mesmo orcamento, dividido de todas as maneiras.
dividido = partilha.partilha(dobra, (ORCAMENTO,), FRACOES, inicio=QUANDO, horizonte=HORIZONTE)
linhas = []
for f in FRACOES:
    cn, cc = partilha.ciclos(f * ORCAMENTO, (1.0 - f) * ORCAMENTO, HORIZONTE)
    linhas.append({"fracao para a escala": f, "ciclo escala": cn, "ciclo forma": cc,
                   "absorvido": dividido["media"][(ORCAMENTO, f)],
                   "dispersao": dividido["dispersao"][(ORCAMENTO, f)]})
tabela = pd.DataFrame(linhas).set_index("fracao para a escala")
print("orcamento de %d atualizacoes em %d dias" % (ORCAMENTO, HORIZONTE))
print(tabela.round(6).to_string())
melhor = tabela["absorvido"].idxmin()
meio = tabela.loc[0.5, "absorvido"]
print()
print("melhor fracao: %.2f (absorvido %.6f) | meio a meio %.6f | pior %.6f"
      % (melhor, tabela["absorvido"].min(), meio, tabela["absorvido"].max()))


orcamento de 60 atualizacoes em 500 dias
                      ciclo escala  ciclo forma  absorvido  dispersao
fracao para a escala                                                 
0.00                           500            8   0.003376   0.001218
0.25                            33           11   0.000529   0.000124
0.50                            17           17   0.000479   0.000106
0.75                            11           33   0.000441   0.000088
0.90                             9           83   0.000425   0.000090
1.00                             8          500   0.000445   0.000151

melhor fracao: 0.90 (absorvido 0.000425) | meio a meio 0.000479 | pior 0.003376


In [4]:
# O orcamento apertando: quando a partilha passa a morder.
linhas = []
for w in ORCAMENTOS:
    v = []
    for f in (0.5, 0.9):
        cn, cc = partilha.ciclos(f * w, (1.0 - f) * w, HORIZONTE)
        v.append(partilha.perdas(dobra, cn, cc, inicio=QUANDO, horizonte=HORIZONTE)[0])
    linhas.append({"atualizacoes": w, "meio a meio": v[0], "quase toda na escala": v[1],
                   "ganho_pct": 100.0 * (v[0] - v[1]) / v[0]})
orcamento_tabela = pd.DataFrame(linhas).set_index("atualizacoes")
print(orcamento_tabela.round(6).to_string())
print()
sobra = partilha.perdas(dobra, 1, 1, inicio=QUANDO, horizonte=HORIZONTE)
print("tudo atualizado todo dia: %.6f, com %d atualizacoes de cada ingrediente"
      % (sobra[0], HORIZONTE))


              meio a meio  quase toda na escala  ganho_pct
atualizacoes                                              
5                0.001551              0.001087  29.938725
10               0.000988              0.000722  26.897469
20               0.000665              0.000519  21.986716
40               0.000514              0.000455  11.457867
60               0.000479              0.000425  11.294479
120              0.000429              0.000403   5.989090
250              0.000406              0.000399   1.762920

tudo atualizado todo dia: 0.000397, com 500 atualizacoes de cada ingrediente


## O que os numeros dizem

A escala e a forma nao tem a mesma pressa, e a diferenca e grande. Atualizar a escala a cada
duzentos e cinquenta dias custa caro; atualizar a forma com a mesma lentidao custa quase nada,
porque padronizar ja tirou dela o que a mudanca carregava.

Com o orcamento dividido, o melhor nao e o meio a meio: quase tudo vai para a escala e uma fatia
pequena para a forma. Dar tudo a escala tambem piora, porque a forma fica velha e a barreira
escorrega de nivel. O otimo e interior, e e assimetrico.

E a partilha so morde quando o orcamento e pobre. Com muitas atualizacoes, dividir de qualquer
maneira da quase o mesmo; com poucas, a escolha vale um quarto da perda. A competicao que a
pergunta supoe existe, mas so aparece no aperto.

O aperto, medido contra a sobra, pesa pouco e nem sempre pesa. Comparados par a par nos mesmos
vinte mundos, o desenho de sessenta atualizacoes entrega mais perda em catorze deles e menos em
seis, e o desvio dessa diferenca entre mundos e maior que a media dela. A afirmacao que a media
sozinha autoriza e mais fraca do que a media sugere --- e comparar as dispersoes de cada desenho
isoladamente teria escondido isso, porque os dois sao medidos nos mesmos mundos.


In [5]:
# Figura 1: as duas pressas, com a dispersao desenhada.
fig, eixo = plt.subplots(figsize=(8.6, 4.4))
raios = 1.959963985 / np.sqrt(SEMENTES)
for coluna, dp, cor, rotulo in (("escala", "escala_dp", "#1f4e79", "a escala (o nível)"),
                                ("forma", "forma_dp", "#b03a2e", "a forma (o corte padronizado)")):
    eixo.errorbar(prensas.index, prensas[coluna], yerr=raios * prensas[dp], marker="o", capsize=3,
                  color=cor, lw=1.6, label=rotulo)
eixo.axhline(piso[0], color="#555555", ls=":", lw=1.2, label="mundo que não muda")
eixo.set_xscale("log")
eixo.set_xticks(CICLOS)
eixo.set_xticklabels([str(c) for c in CICLOS], fontsize=9)
eixo.set_xlabel("dias entre duas atualizações do ingrediente")
eixo.set_ylabel("perda absorvida por dia")
eixo.legend(frameon=False, fontsize=9)
eixo.grid(alpha=0.25)
fig.tight_layout()
graficos.salvar(fig, "E11_partilha_do_relogio", 1)
plt.close(fig)
print("lento demais: escala %.6f | forma %.6f" % (prensas.loc[250, "escala"], prensas.loc[250, "forma"]))


lento demais: escala 0.001807 | forma 0.000383


In [6]:
# Figura 2: a partilha no aperto, e o ganho contra o orcamento.
fig, (esq, dir_) = plt.subplots(1, 2, figsize=(9.8, 4.2))
esq.errorbar(tabela.index, tabela["absorvido"], yerr=raios * tabela["dispersao"], marker="o",
             capsize=3, color="#1f4e79", lw=1.6)
esq.scatter([melhor], [tabela["absorvido"].min()], s=140, marker="*", color="#b03a2e", zorder=5,
            label="a melhor partilha")
esq.axvline(0.5, color="#555555", ls="--", lw=1.2, label="meio a meio")
esq.set_xlabel("fração do orçamento que vai para a escala")
esq.set_ylabel("perda absorvida por dia")
esq.legend(frameon=False, fontsize=8)
esq.grid(alpha=0.25)
dir_.plot(orcamento_tabela.index, orcamento_tabela["ganho_pct"], marker="o", color="#2e7d32", lw=1.6)
dir_.set_xscale("log")
dir_.set_xticks(list(ORCAMENTOS))
dir_.set_xticklabels([str(w) for w in ORCAMENTOS], fontsize=9)
dir_.set_xlabel("atualizações no horizonte")
dir_.set_ylabel("ganho da melhor partilha (%)")
dir_.grid(alpha=0.25)
fig.tight_layout()
graficos.salvar(fig, "E11_partilha_do_relogio", 2)
plt.close(fig)
print("ganho no aperto: %.1f%% com %d atualizacoes | %.1f%% com %d"
      % (orcamento_tabela.loc[ORCAMENTOS[0], "ganho_pct"], ORCAMENTOS[0],
         orcamento_tabela.loc[ORCAMENTOS[-1], "ganho_pct"], ORCAMENTOS[-1]))


ganho no aperto: 29.9% com 5 atualizacoes | 1.8% com 250


## Leitura visual das figuras

Feita nesta sessão abrindo os .png com a ponte de visão (AGENTS.md §9). Observação, não número.

**Figura 1.** Duas curvas e uma linha de base. A vermelha, a forma, é praticamente uma reta deitada
sobre a linha pontilhada do mundo que não muda: atravessa a figura inteira sem subir. A azul, a
escala, começa colada na vermelha nos três primeiros pontos e depois sobe, e as barras de erro
abrem junto — a curva que sobe é também a curva que fica incerta. O que o eixo engana: o
horizontal é logarítmico, de modo que o trecho em que a escala dispara ocupa menos da metade da
largura, e a distância entre as duas últimas marcas vale mais dias do que a distância entre as
duas primeiras, embora as duas pareçam iguais.

**Figura 2.** À esquerda, a perda cai de um penhasco entre zero e um quarto e depois fica rasa: a
estrela do melhor desenho está quase encostada na borda direita, e a linha tracejada do meio a
meio cai já no trecho plano, de modo que a figura mostra sozinha que o palpite simétrico quase
não custa e que o que custa é esquecer a escala. À direita, o ganho desce de forma quase monótona
e o que o eixo engana é a unidade: o ganho é uma fração da perda, e a perda cai junto com o
orçamento, de modo que os dois painéis não se leem na mesma escala.


In [7]:
# O aperto conta o que a sobra dava de graca? Os dois desenhos sao medidos nos MESMOS
# mundos, entao a comparacao honesta e par a par: a dispersao de cada desenho sozinho
# nao diz se a diferenca entre eles sobrevive ao sorteio.
cn_sobra, cc_sobra = partilha.ciclos(ORCAMENTO * melhor, ORCAMENTO * (1.0 - melhor), HORIZONTE)
por_mundo = np.array([
    partilha.absorvido(s, cn_sobra, cc_sobra, inicio=QUANDO, horizonte=HORIZONTE)
    - partilha.absorvido(s, 1, 1, inicio=QUANDO, horizonte=HORIZONTE)
    for s in dobra
])
aperto_por_mundo = float(por_mundo.mean())
aperto_dispersao = float(por_mundo.std(ddof=1))
print('aperto, mundo a mundo: media %.8f | dispersao %.8f | media/dispersao %.2f | mundos em que o aperto perde %.2f'
      % (aperto_por_mundo, aperto_dispersao, aperto_por_mundo / aperto_dispersao,
         float((por_mundo > 0).mean())))


aperto, mundo a mundo: media 0.00002820 | dispersao 0.00005052 | media/dispersao 0.56 | mundos em que o aperto perde 0.70


In [8]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
resultado = {
    "partilha_sementes": int(SEMENTES),
    "partilha_dias_de_serie": int(DIAS),
    "partilha_horizonte": int(HORIZONTE),
    "partilha_janela_escala": int(partilha.JANELA_NIVEL),
    "partilha_janela_forma": int(partilha.JANELA_CORTE),
    "partilha_posto": int(partilha.POSTO),
    "partilha_orcamento": int(ORCAMENTO),
    "partilha_piso_parado": float(piso[0]),
    "partilha_piso_parado_dispersao": float(piso[1]),
    "partilha_escala_dia": float(prensas.loc[1, "escala"]),
    "partilha_escala_lenta": float(prensas.loc[250, "escala"]),
    "partilha_forma_dia": float(prensas.loc[1, "forma"]),
    "partilha_forma_lenta": float(prensas.loc[250, "forma"]),
    "partilha_razao_escala": float(prensas.loc[250, "escala"] / prensas.loc[1, "escala"]),
    "partilha_razao_forma": float(prensas.loc[250, "forma"] / prensas.loc[1, "forma"]),
    "partilha_melhor_fracao": float(melhor),
    "partilha_melhor": float(tabela["absorvido"].min()),
    "partilha_meio_a_meio": float(meio),
    "partilha_sem_atualizar_escala": float(tabela.loc[0.0, "absorvido"]),
    "partilha_ganho_meio_a_meio_pct": float(100.0 * (meio - tabela["absorvido"].min()) / meio),
    "partilha_sobra": float(sobra[0]),
    "partilha_sobra_dispersao": float(sobra[1]),
    "partilha_sobra_atualizacoes": int(HORIZONTE),
    "partilha_aperto_sobre_tudo_pct": float(100.0 * (tabela["absorvido"].min() - sobra[0]) / sobra[0]),
    "partilha_aperto_por_mundo": float(aperto_por_mundo),
    "partilha_aperto_por_mundo_dispersao": float(aperto_dispersao),
    "partilha_aperto_mundos_que_perdem_contagem": int((por_mundo > 0).sum()),
    "partilha_aperto_mundos_que_ganham_contagem": int((por_mundo < 0).sum()),
}
for w in (5, 20, 60, 250):
    nome = {5: "cinco", 20: "vinte", 60: "sessenta", 250: "duzentos_e_cinquenta"}[w]
    resultado["partilha_ganho_%s_pct" % nome] = float(orcamento_tabela.loc[w, "ganho_pct"])
    resultado["partilha_quase_toda_%s" % nome] = float(orcamento_tabela.loc[w, "quase toda na escala"])

caminho = Path("lab/resultados/E11_partilha_do_relogio.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True),
                   encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))


lab/resultados/E11_partilha_do_relogio.json gravado | 36 grandezas
